In [ ]:
import pandas as pd
import datetime
import python_files.functions_barcelona as fb
import python_files.paths_mapping as pm
# !pip install holidays_es
try:
    from holidays_es import Province, HolidaySpain
    festes=True
except:
    festes=False
    

In [ ]:
last_year = 2025
accidents=pd.read_csv(pm.FILES_PATH / f'accidents_weather{last_year}.csv',date_format='%Y/%m/%d %H:%M:%S')
accidents['datetime']=pd.to_datetime(accidents.datetime, utc=True)
#fixing street names
accidents['street_name']=accidents.street_name.str.strip()
#casting some columns to int
for col in ['year','day','hour']:
    accidents[col]=accidents[col].astype(int)
accidents.head()

<a id='start'></a>
# Feature Engineering
 1. [**Binning streets and neighborhoods and creating a count of accidents.**](#binning) We should be careful with this one. I created a function that does the count with the train and applies it to the test. I have assumed that 2022 and 2023 are the test. I have done the count (street_count) and made 5 bins (street_bins). The same applies to neighborhoods.
 2. [**Creating a holiday column.**](#holidays) Assign 1 if holiday, 0 if previoous to holiday and -1 if non a holiday
 3. [**Identifying crossroads.**](#crossroads) It might be interesting evaluate if the fact of the accident taking place in a cross road is relevant.
 4. [**Binning ages.**](#binning) We will make 5 categories within ages

<a id='binning'></a>
## 1. **Binning streets and neighborhoods and creating a count of accidents**.

   
[Back](#start)

In [ ]:
def binning_fields(df,field, last_year_train,num_bins=5):
    """binning the streets for train and adding it
    without leakage to test"""
    train=df[df.year<=last_year_train]
    df[field +'_bins']=pd.qcut(
    df[field].map(
        train[field].value_counts().to_dict()),
                                  num_bins,
                                  labels=False)
    df[field+'_count']=df[field].map(train[field].value_counts().to_dict())
    df[field+'_count'].fillna(0,inplace=True)
    df[field +'_bins'].fillna(0,inplace=True)
    return df
accidents=binning_fields(accidents,'street_name',2021)

In [ ]:
accidents.groupby('street_name').street_name_count.mean().sort_values(ascending=False).iloc[:10]

In [ ]:
accidents.groupby('street_name_bins')['street_name_count'].mean()

In [ ]:
accidents=binning_fields(accidents,'neighborhood',2021,num_bins=3)

In [ ]:
accidents.groupby('neighborhood').neighborhood_count.mean().sort_values(ascending=False).iloc[:5]

In [ ]:
accidents.groupby('neighborhood_bins')['neighborhood_count'].mean()

<a id='holidays'></a>
## 2. **Creating a holiday column.**

   
[Back](#start)

In [ ]:
#Gathering all holidays either Spanish, Catalan or local from 2010 to 2024
holidays_bcn=[]
for year in range(2010,2024):
    holidays=HolidaySpain(province=Province.BARCELONA,year=year)
    for holiday in holidays.national:
        holidays_bcn.append(holiday.date)
        
    for holiday in holidays.regional:
        holidays_bcn.append(holiday.date)
    for holiday in holidays.local:
        holidays_bcn.append(holiday.date)
#defining a function to determine if a given date (TimeStamp) is a holiday, holiday eve or none.
def adding_holiday(date_time):
    date_time=datetime.datetime.date(date_time)
    if date_time in holidays_bcn:
        return 1
    elif date_time +datetime.timedelta(days=1) in holidays_bcn:
        return 0
    else:
        return -1
#generating the new column

accidents['is_holiday']=accidents['datetime'].apply(adding_holiday)

<a id='crossroads'></a>
## 3. **Identifying accidents that occur in a crossing.**

   
[Back](#start)

In [ ]:
#2. Identifying accidents that occur in a crossing
accidents['crossing_street']=[1 if '/' in x else 0 for x in accidents.street_name]

In [ ]:
##Adding target
def creating_target(row):
    return 1 if row['num_deaths']+row['num_severly_injured']>0 else 0
accidents['target']=accidents.apply(creating_target,axis=1)


In [ ]:
##Saving the file with new features
accidents.to_csv('./data/accidents_weather_eng_2023.csv',index=False)